# Peta Sumber Data & Fitur Final — ISPA Jawa Timur 2024

Primary key: `data/reference/dim_kabupaten_jatim.csv` — 38 baris (29 kabupaten + 9 kota), kode BPS 35xx.

| Field | Value |
|---|---|
| Periode | 2024 |
| Unit analisis | Kabupaten/kota (n = 38) |
| Desain | Clustering (blok A) → label → klasifikasi (blok B) |
| Terakhir diperbarui | 2026-11-25 |

> **Peringatan join:** nama wilayah tidak unik. "Blitar", "Kediri", "Malang", "Probolinggo", "Pasuruan", "Mojokerto", dan "Madiun" masing-masing ada dua. Join wajib memakai `kode_bps`.

## 0. Fitur Final — 6 Variabel

### Blok A — pembentuk label (clustering)

| Kode | Variabel | Rumus / Definisi | Satuan | Sumber | Status |
|---|---|---|---|---|---|
| `ir_ispa` | Incidence Rate ISPA | (jumlah kasus ISPA ÷ jumlah penduduk) × 1.000 | kasus/1.000 penduduk | Profil Kesehatan Jatim 2024 | **Belum ada** |
| `rate_pneu` | Rate pneumonia balita | (kasus pneumonia balita ÷ jumlah balita) × 1.000 | kasus/1.000 balita | Profil Kesehatan Jatim 2024 | Opsional |

### Blok B — prediktor klasifikasi (kontekstual)

| Kode | Variabel | Definisi | Satuan | Sumber | Status |
|---|---|---|---|---|---|
| `iku` | Indeks Kualitas Udara | Indeks komposit 0–100 | poin | Satu Data Jatim | **Siap — 38/38** |
| `curah_hujan` | Curah hujan tahunan 2024 | Total curah hujan setahun | mm/tahun | BPS, Jatim Dalam Angka 2025 | Belum ada |
| `kepadatan` | Kepadatan penduduk | Penduduk ÷ luas wilayah | jiwa/km² | BPS, Jatim Dalam Angka 2025 | Belum ada |
| `sanitasi` | Akses sanitasi layak | % rumah tangga dengan sanitasi layak | % | Profil Kesehatan Jatim 2024 | Belum ada |

**Total 6 variabel, n = 38.** Rasio 6 variabel untuk 38 observasi masih wajar; di atas 8 variabel jarak antar-observasi mulai tidak bermakna.

### 0.1 Alasan pemilihan

| Keputusan | Alasan |
|---|---|
| Blok A hanya berisi outcome | Kalau fitur kontekstual ikut membentuk label, klasifikasi di blok B jadi sirkular — model hanya menghafal |
| Dua variabel di blok A, bukan satu | `ir_ispa` mengukur beban semua umur, `rate_pneu` mengukur beban balita. Dua dimensi membuat clustering punya dasar lebih dari sekadar urutan satu angka |
| IKU tetap dipakai walau arahnya terbalik | Variansnya nyata (71,91–94,84; sd 5,03), jadi informatif. Arah dijelaskan saat interpretasi |
| Curah hujan dipakai tahunan, bukan bulanan | ISPA diamati setahun; curah hujan satu bulan tidak mewakili paparan |
| Kepadatan dipakai, bukan jumlah penduduk | Jumlah penduduk akan mengelompokkan wilayah berdasarkan besar-kecilnya, bukan risikonya |

### 0.2 Variabel yang dibuang

| Variabel | Alasan |
|---|---|
| Paparan asap rokok | Hanya tersedia level provinsi (Riskesdas 2018, SKI 2023). Bila dipaksakan, varians = 0 dan z-score tidak terdefinisi |
| Gejala klinis per pasien (RR, batuk, pilek, demam, dll.) | Data rekam medis, tidak publik, dan unit analisisnya pasien — bukan wilayah |

## 1. Status per Variabel

| No | Variabel | Blok | File | Cakupan | Status |
|---|---|---|---|---|---|
| 1 | Indeks Kualitas Udara | B | `indeks-kualitas-udara-tahun-2024.xlsx` | 38/38 | **BERSIH — siap pakai** |
| 2 | Curah hujan | B | `jmlh_crh_hjn_rrt_blnn...csv` | 31/38, 1 bulan | **PERLU DATA ULANG** |
| 3 | Kepadatan penduduk | B | — | — | Belum ada |
| 4 | Sanitasi rumah tangga | B | — | — | Belum ada |
| 5 | Incidence Rate ISPA | A | — | — | **Belum ada — wajib** |
| 6 | Rate pneumonia balita | A | — | — | Opsional |
| — | Paparan asap rokok | — | — | — | **Dibuang** |

## 1A. Hasil Eksplorasi BPS API (domain 3500)

Diuji 2026-11-25 memakai `BPS_API_KEY` dari `.env`. Domain 3500 = Provinsi Jawa Timur.

| Endpoint | Hasil |
|---|---|
| `list/model/var/domain/3500` | 413 variabel |
| `list/model/statictable/domain/3500` | 2.456 tabel |
| `list/model/publication/domain/3500` | 1.359 publikasi |

### Cakupan tahun per variabel

| Variabel | var_id | Level | Tahun terbaru di API |
|---|---|---|---|
| Persentase rumah tangga akses sanitasi layak | 486 | Kabupaten/kota (39) | **2019** |
| Jumlah penduduk menurut kab/kota | 375 | Kabupaten/kota (39) | **2023** |
| Curah hujan | 96 | **Provinsi, per bulan** | 2024 (bulanan, bukan kabupaten) |
| Hari hujan | 98 | **Provinsi, per bulan** | 2024 (bulanan, bukan kabupaten) |
| Kepadatan penduduk | 426 | **Provinsi se-Indonesia** | — |

### Tabel statistik (statictable) terbaru

| Kebutuhan | table_id | Judul | Cakupan tahun |
|---|---|---|---|
| Kepadatan per kab/kota | 2635 | Distribusi Persentase Penduduk dan Kepadatan Penduduk Menurut Kabupaten/Kota | **2020 dan 2022** |
| Sanitasi per kab/kota | 3019 | Persentase Rumah Tangga dengan Akses Sanitasi Layak Menurut Kabupaten/Kota | **2017–2022** |
| Pneumonia per kab/kota | 3001 | Jumlah Penyakit Malaria, TB Paru, Pneumonia, Kusta Menurut Kabupaten/Kota | **2022** |
| ISPA | — | **tidak ada** | — |
| Curah hujan per kab/kota | — | **tidak ada** (hanya Stasiun Juanda) | — |

### Kesimpulan

**API BPS tidak menyediakan data 2024 untuk kebutuhan kita.** Data per kabupaten paling baru adalah 2022 (kepadatan, sanitasi, pneumonia), dan ISPA tidak ada sama sekali.

Karena itu data 2024 hanya bisa diambil dari **berkas publikasi**, bukan dari API:

| Publikasi | pub_id | Ukuran | Unduh |
|---|---|---|---|
| Provinsi Jawa Timur Dalam Angka 2025 | `5ae8b994b1bebd8cbc2e0781` | 28,27 MB | `scripts/bps_download_pub.py` |
| Profil Kesehatan Jatim 2024 | — | — | manual, situs Dinkes |

Catatan teknis: server `download.php` BPS **tidak mengirim Content-Length** dan **tidak mendukung Range**, sehingga unduhan bisa terputus tanpa error. Skrip memverifikasi penanda `%%EOF` dan mengulang bila berkas tidak lengkap.

## 2. Indeks Kualitas Udara — VERIFIKASI SELESAI

| Pemeriksaan | Hasil |
|---|---|
| Baris awal | 39 (38 kabupaten/kota + 1 baris agregat Provinsi Jawa Timur) |
| Baris agregat | Dihapus (`kode_kabupaten_kota = "-"`) |
| Hasil join ke dim | **38 baris, 0 kode tidak cocok, 0 duplikat** |
| Nilai hilang | 0 |
| Rentang | 71,91 – 94,84 poin; rata-rata 88,09; simpangan baku 5,03 |

| Wilayah | IKU |
|---|---|
| Sidoarjo (terendah) | 71,91 |
| Kota Mojokerto | 79,22 |
| Bangkalan | 80,23 |
| Ponorogo (tertinggi) | 94,09 |
| Kab. Kediri | 94,31 |
| Trenggalek | 94,84 |

**Arah:** IKU 0–100, **semakin tinggi semakin baik**. Berlawanan dengan fitur risiko lain, sehingga perlu diperhatikan saat membaca profil cluster.

**Keluaran:** `data/processed/iku_jatim_2024.csv`.

## 2A. Profil Kesehatan Jatim — Struktur Tabel (dari edisi 2025)

Berkas yang diterima: `105__1784015649_profil-kesehatan-2025-14072026.pdf` — 439 halaman, 4,60 MB.

> **Peringatan tahun:** tabel di dalamnya berjudul **"TAHUN 2025"**. Edisi ini memuat data 2025, bukan 2024. Struktur tabelnya tetap dicatat karena edisi 2024 memakai format yang sama.

### Tabel yang relevan

| Tabel | Halaman | Isi | Kolom penting |
|---|---|---|---|
| Tabel 61 | 409 | Realisasi penemuan pneumonia balita per kabupaten/kota | jumlah balita, perkiraan pneumonia, penemuan pneumonia (L/P/L+P), pneumonia berat, batuk bukan pneumonia |
| Tabel 84 | 435 | Akses sanitasi per kabupaten/kota | jumlah KK, KK pengguna, akses sanitasi aman %, layak sendiri %, layak bersama %, belum layak %, BABS tertutup %, BABS terbuka % |
| Tabel 8–10 | 339–341 | 10 penyakit / kematian terbanyak | **hanya level provinsi**, berbasis ICD-10 |

### Tidak ada ISPA per kabupaten

Kata "ISPA" muncul di 8 halaman, tetapi seluruhnya berupa narasi. Tabel penyakit terbanyak disajikan hanya sampai level provinsi dan memakai kode ICD-10 (J06.9 "Penyakit sistem pernapasan"), bukan kategori ISPA.

**Konsekuensi:** `ir_ispa` per kabupaten/kota kemungkinan besar tidak tersedia di publikasi mana pun. Variabel outcome yang realistis adalah **`rate_pneu`** (pneumonia balita per 1.000 balita).

### Rumus outcome yang bisa dibentuk dari Tabel 61

```
rate_pneu = (realisasi penemuan pneumonia balita L+P ÷ jumlah balita) x 1.000
```

Bila ingin dua dimensi untuk clustering, tambahan yang tersedia dari tabel yang sama:

```
prop_pneu_berat = (pneumonia berat L+P ÷ total penemuan pneumonia L+P) x 100
```

## 3. Curah Hujan — EMPAT MASALAH

File: `data/raw/data_curah_hujan_mentah/jmlh_crh_hjn_rrt_blnn_d_kbptnkt_brdsrkn_wlyh_sng.csv`
Sumber: Dinas PU Sumber Daya Air Provinsi Jawa Timur (Satu Data Jatim).

| No | Masalah | Angka | Dampak |
|---|---|---|---|
| 1 | Hanya **satu bulan** (Juli 2024) | `periode_update` = 2024-07 saja, 76 baris | Juli musim kemarau; tidak mewakili setahun |
| 2 | **7 kabupaten hilang** | 31 dari 38 wilayah | 18% observasi kosong |
| 3 | **7 kota punya 4 baris** | Kediri, Blitar, Malang, Probolinggo, Pasuruan, Mojokerto, Madiun | Satu wilayah jadi dua nilai |
| 4 | Nilai duplikat bertentangan | Kota Kediri 27,25 vs 8,66 mm | Tidak bisa dipilih sembarangan |

**Pola mencurigakan:** 7 kabupaten yang hilang persis sama dengan 7 kota yang terduplikasi — indikasi kode wilayah tertukar di sumber.

### 3.1 Rekomendasi

Ambil tabel curah hujan tahunan dari **Provinsi Jawa Timur Dalam Angka 2025** (BPS): 38 wilayah lengkap, satu angka per tahun, satu file bersama kepadatan penduduk.

File Satu Data yang ada tetap disimpan sebagai bukti pemeriksaan kualitas (bahan F.2.1), bukan sebagai sumber fitur.

## 4. Yang Masih Harus Diunduh

| Prioritas | Variabel | Publikasi | Tabel yang dicari |
|---|---|---|---|
| 1 | `ir_ispa` + `rate_pneu` | Profil Kesehatan Provinsi Jawa Timur 2024 | jumlah kasus ISPA dan pneumonia balita per kabupaten/kota |
| 2 | `curah_hujan` + `kepadatan` | Provinsi Jawa Timur Dalam Angka 2025 | curah hujan tahunan; luas wilayah & kepadatan penduduk |
| 3 | `sanitasi` | Profil Kesehatan Provinsi Jawa Timur 2024 | % rumah tangga akses sanitasi layak |

Ketiganya bisa didapat dari **dua publikasi saja** — Profil Kesehatan untuk data kesehatan, Dalam Angka untuk data wilayah.

### 4.1 Catatan penamaan publikasi

BPS menamai publikasi dengan **tahun terbit**, bukan tahun data.

| Publikasi | Isi data |
|---|---|
| Provinsi Jawa Timur Dalam Angka 2024 | 2023 |
| Provinsi Jawa Timur Dalam Angka 2025 | **2024** |
| Profil Kesehatan Provinsi Jawa Timur 2024 | **2024** |

Jadi untuk data 2024, yang diunduh adalah **Dalam Angka 2025**.

Pengecualian: sebagian tabel dapat tertinggal satu tahun karena keterlambatan data daerah. Setelah PDF diunduh, periksa kolom tahun pada tabel curah hujan. Bila masih 2023, ambil dari BMKG atau agregasi 12 bulan Satu Data Jatim.



## 5. Rencana Join

```
dim_kabupaten_jatim.csv (38, kode_bps)
   ├── iku_jatim_2024            ✓ 38/38
   ├── curah_hujan_2024          ← BPS Dalam Angka
   ├── kepadatan_2024            ← BPS Dalam Angka
   ├── sanitasi_2024             ← Profil Kesehatan
   ├── ir_ispa_2024              ← Profil Kesehatan
   └── rate_pneu_2024            ← Profil Kesehatan
                ▼
        master_38x8.csv  (data/processed/)
```

Aturan:

1. File mentah di `data/raw/`, tidak pernah diedit.
2. Setiap sumber diberi `kode_bps` sebelum join; nama hanya untuk verifikasi manual.
3. Setelah join, jumlah baris **wajib tetap 38**. Bertambah = duplikat kode; berkurang = kode tidak cocok.
4. Setiap ketidakcocokan dicatat, tidak dibuang diam-diam.

## 6. Implikasi ke Tahap Klasifikasi

| Aspek | Konsekuensi |
|---|---|
| n = 38, 4 prediktor | Model harus sederhana: decision tree, random forest kedalaman terbatas, KNN, atau naive Bayes |
| 3 kelas | Sekitar 12–13 observasi per kelas — split latih/uji harus stratified |
| Label dari 2 variabel, prediktor 4 variabel lain | Akurasi wajar bila sedang (50–70%). Akurasi rendah bukan kegagalan, melainkan temuan: faktor lingkungan saja belum cukup menjelaskan risiko ISPA |
| Standardisasi | Wajib untuk KNN dan clustering; tidak wajib untuk decision tree/random forest |